In [ ]:
import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use("dark_background")

# 1 Attention: reweighting by relevance instead of by distance

Every mechanism in this notebook is one you could import in a single line. You build them
by hand once, here, because attention is the operation the rest of modern machine learning
is assembled from, and "it reweights the sequence" is not an explanation you can debug.

The route: a Gaussian filter reweights a signal by *distance*, which is the wrong prior
for language. Attention reweights by *relevance*, learned from the data. Then positional
encoding puts the distance information back — deliberately, as a separate ingredient —
and the two together are a transformer block.

# Reweighing based on proximity

We get some simple timeseries

In [ ]:
t = torch.linspace(0, 4 * torch.pi, 100)


def f(t):
    noise = torch.randn(100) * 0.1
    return torch.sin(t) + noise


x = f(t)

And let's create a filter that smooths the signal by taking in the neighbouring signals

In [ ]:
# initialize a convolution
conv = torch.nn.Conv1d(1, 1, 8)

# build a gaussian kernel
n = torch.distributions.normal.Normal(0, 1)
v = torch.arange(-4, 4)
gaussian = torch.exp(n.log_prob(v))[None, None, :]

# replace the random weights
d = conv.state_dict()
d["weight"] = gaussian
conv.load_state_dict(d)
x_ = conv(x[None, None, :])[0][0]

We can plot everything like this:

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].plot(t, x)
ax[0].set_title("original")

ax[1].scatter([*range(8)], gaussian[0][0])
ax[1].set_title("gaussian filter")

ax[2].plot(x_.detach())
ax[2].set_title("smoothed")

So, you see a noisy signal. The gaussian filter takes adds more of the close values, and less of the values further away. The filter works on the direct neighborhood.

# Text

Now, we consider text.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer

tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
trainer = WordLevelTrainer(special_tokens=["[UNK]"])
tokenizer.pre_tokenizer = (
    Whitespace()
)  # This pre-tokenizer simply splits using the following regex: \w+|[^\w\s]+

In [ ]:
corpus = ["ik zit op de bank", "ik werk bij de bank", "bij de bank is het heel druk"]

In [ ]:
tokenizer.train_from_iterator(corpus, trainer=trainer)
tokenizer.get_vocab()

Now, the problem with text is: it is not necessarily the words that are close, that have the most impact.

We see three sentences, max seven words, so dimensions are (3,7)

In [ ]:
x = tokenizer.encode(corpus[0])
x.ids

Our sentences are being encoded, and the word "bank" gets the integer 6 assigned. However, the meaning of this word is not the same because of the context... If we make an embedding:

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

vocab_len = tokenizer.get_vocab_size()

emb = nn.Embedding(num_embeddings=vocab_len, embedding_dim=4, padding_idx=0)

tokenized_corpus = pad_sequence(
    [torch.tensor(tokenizer.encode(s).ids) for s in corpus], batch_first=True
)

embeddings = emb(tokenized_corpus)

embeddings, embeddings.shape

We have added a dimensionality of 4 to every word. So now we have (3, 7, 4).
You can see that the word "bank" gets exactly the same vector, as expected...

In [ ]:
bank1 = embeddings[0][4]
bank2 = embeddings[1][4]
bank1, bank2

# Attention
Now we will start with the attention mechanism.
We need a key, query and value. Because we use self attention, these are just clones.

In [ ]:
key = embeddings.detach().clone()
query = embeddings.detach().clone()
values = embeddings.detach().clone()
key.shape

We have 4 features

In [ ]:
d_features = torch.tensor(query.shape[-1])
d_features

And with this, we can calculate $$\frac{(QK^T)}{\sqrt{d}}$$

In [ ]:
dots = torch.bmm(query, key.transpose(1, 2)) / torch.sqrt(d_features)
dots.shape

This gives us a shape of (3, 7, 7):
for every sentence, we have for every word, weights how we want to mix in every other word. So this last part always has a shape (sequence, sequence)

We obtain the weights with a softmax:

In [ ]:
weights = nn.Softmax(dim=-1)(dots)

weights[0]

and finally we can do a matrix-multiplication with the values:

$$attention = softmax\left(\frac{(QK^T)}{\sqrt{d}}\right)V$$

In [ ]:
activations = torch.bmm(weights, values)
activations.shape, embeddings.shape

note how we end up with exactly the same size: 3 sentences, max 7 words, but now every word has 4 dimensions that are reweighted by the other words in the sentence, regardless of the distance, but mainly depending on the semantics (meaning) of every word as encoded in de embedding.

In [ ]:
bank1 = activations[0][4]
bank2 = activations[1][4]
bank1, bank2

Now, the vector for the word bank has been "mixed" with all the other words in the sentence, and they are different!

torch has a multihead attention implemented. With that, we can add a mask to cover the padding.

In [ ]:
mask = tokenized_corpus == 0

In [ ]:
multihead = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)
attn, attn_w = multihead(query, key, values, key_padding_mask=mask)

In [ ]:
attn.shape

It is possible to visualize the weights. In this case, this is untrained.
What you expect is that after training the vector for the word "bank" should be mostly mixed with the word "zit" (sit) to make more sense.

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 10))
labels = corpus[0].split()
labels = labels + ["PAD", "PAD"]

plot = sns.heatmap(attn_w[0].detach().numpy())

plot.set_xticklabels(labels)
plot.set_yticklabels(labels);

## 1.6 Positional encoding, and the block it completes

Attention as built above is **permutation-equivariant**: shuffle the words and the output
shuffles with them, unchanged. For a bag of features that is fine. For a sentence it is a
catastrophe — "the dog bit the man" and "the man bit the dog" produce identical
representations.

So position has to be added back explicitly. That is what the rest of this notebook does,
and it is worth noticing that this is the *opposite* choice from the convolution you
started with: a convolution has locality baked in and cannot easily look far away;
attention can look anywhere and has to be *told* where things are.

# Positional Encoding
A final ingredient for the transformer architecture is positional encoding.

With Attention we loose all sense of location in a sentence. A way around this is to add positional encoding.

The idea is to add multiple sine waves.
One sinewave will have a very fast frequency, another sinewave will have a slow frequency.
Note that this is exactly how our time-system works: minutes have a fast frequency, hours a slower one, days of the week even slower etc.

By combining a lot of sinewaves the model is able to figure out something about the location in time, relative to each other. 

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn, Tensor
import math


class PositionalEncoding(nn.Module):
    pe: Tensor

    def __init__(self, d_model: int, dropout: float = 0.1, max_seq_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create matrix of [SeqLen, HiddenDim] representing the positional encoding for max_len inputs
        position = torch.arange(max_seq_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(1, max_seq_len, d_model)
        # batch, seq_len, d_model
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)

        # register_buffer => Tensor which is not a parameter, but should be part of the modules state.
        # Used for tensors that need to be on the same device as the module.
        # persistent=False tells PyTorch to not add the buffer to the state dict (e.g. when we save the model)
        self.register_buffer("pe", pe)

    def forward(self, x: Tensor) -> Tensor:
        """
        Arguments:
            x: Tensor, shape ``[batch_size, seq_len, embedding_dim]``
        """
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


encod_block = PositionalEncoding(d_model=48, dropout=0.1, max_seq_len=96)
pe = encod_block.pe.squeeze().T.cpu().numpy()

fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111)
ax.imshow(pe, cmap="RdGy", extent=(1, pe.shape[1] + 1, pe.shape[0] + 1, 1))
# fig.colorbar(pos, ax=ax)
ax.set_xlabel("Position in sequence")
ax.set_ylabel("Hidden dimension")
ax.set_title("Positional encoding over hidden dimensions")
ax.set_xticks([1] + [i * 10 for i in range(1, 1 + pe.shape[1] // 10)])
ax.set_yticks([1] + [i * 10 for i in range(1, 1 + pe.shape[0] // 10)])
plt.show()

We can now assemble the encoder block. Encoder-decoder stacks are what translation
models use; the encoder half on its own is enough for classification and regression,
which is what the rest of this lesson needs.

The block is four parts, and you have met three of them: multi-head attention, a
feed-forward network, layer normalisation, and a residual connection around each. The
residual is the same `Skip` idea from lesson 1 — the block learns a *correction* to its
input rather than a replacement.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout):
        super(TransformerBlock, self).__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        self.layer_norm1 = nn.LayerNorm(hidden_size)
        self.layer_norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x):
        identity = x.clone()
        x, _ = self.attention(x, x, x)
        x = self.layer_norm1(x + identity)
        identity = x.clone()
        x = self.ff(x)
        x = self.layer_norm2(x + identity)
        return x


class Transformer(nn.Module):
    def __init__(
        self,
        config: dict,
    ) -> None:
        super().__init__()
        self.pos_encoder = PositionalEncoding(config["hidden"], config["dropout"])

        # Create multiple transformer blocks
        self.transformer_blocks = nn.ModuleList(
            [
                TransformerBlock(
                    config["hidden"], config["num_heads"], config["dropout"]
                )
                for _ in range(config["num_blocks"])
            ]
        )

        self.out = nn.Linear(config["hidden"], config["output"])

    def forward(self, x: Tensor) -> Tensor:
        x = self.pos_encoder(x)

        # Apply multiple transformer blocks
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x)

        x = x.mean(dim=1)
        x = self.out(x)
        return x

Lets test it:

In [ ]:
# create a random torch tensor with batch size 32, sequence length 64 and 128 features
# note that the hidden dimensions of the transformer should be equal to the feature dimensions
X = torch.randn(32, 64, 128)

config = {
    "hidden": 128,
    "dropout": 0.1,
    "num_heads": 4,  # sequence length should be divisible by num_heads
    "num_blocks": 3,
    "output": 2,
}

transformer = Transformer(config)

In [ ]:
output = transformer(X)
output.shape